# Laboratorio CNNs-XAI

Notebook base para ejecutar el flujo del laboratorio usando el codigo del proyecto.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

PROJECT_ROOT

## 1. Exploracion del dataset

In [ ]:
from cnn_xai_lab.config import DATA_DIR, OUTPUTS_DIR
from cnn_xai_lab.data import discover_images, summarize_dataset, save_json, save_sample_mosaic

dataset_frame = discover_images(DATA_DIR)
dataset_summary = summarize_dataset(dataset_frame)
save_json(dataset_summary, OUTPUTS_DIR / "dataset_summary.json")
save_sample_mosaic(dataset_frame, OUTPUTS_DIR / "dataset_mosaic.png")
dataset_summary

## 2. Preprocesamiento y particion

In [ ]:
from cnn_xai_lab.config import BATCH_SIZE, IMAGE_SIZE, SEED
from cnn_xai_lab.data import split_dataframe, summarize_splits, make_tf_dataset

splits = split_dataframe(dataset_frame, seed=SEED)
split_summary = summarize_splits(splits)

train_dataset = make_tf_dataset(splits["train"], IMAGE_SIZE, BATCH_SIZE, training=True, seed=SEED)
val_dataset = make_tf_dataset(splits["val"], IMAGE_SIZE, BATCH_SIZE, training=False, seed=SEED)
test_dataset = make_tf_dataset(splits["test"], IMAGE_SIZE, BATCH_SIZE, training=False, seed=SEED)

split_summary

## 3. Construccion de la CNN y 4. Ajuste de hiperparametros

In [ ]:
from cnn_xai_lab.config import MODELS_DIR
from cnn_xai_lab.training import ExperimentConfig, run_experiments

experiments = [
    ExperimentConfig(
        name="baseline",
        base_filters=32,
        kernel_size=3,
        dense_units=128,
        dropout=0.30,
        learning_rate=1e-3,
    ),
    ExperimentConfig(
        name="wider_k5_dropout",
        base_filters=48,
        kernel_size=5,
        dense_units=128,
        dropout=0.40,
        learning_rate=5e-4,
    ),
]

comparison, best_info = run_experiments(
    experiments=experiments,
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    test_dataset=test_dataset,
    output_dir=OUTPUTS_DIR / "experiments",
    best_model_path=MODELS_DIR / "model.keras",
    image_size=IMAGE_SIZE,
    epochs=10,
)

comparison

## 5. Interpretabilidad con Saliency Map y Grad-CAM

In [ ]:
from PIL import Image
from tensorflow import keras
from cnn_xai_lab.xai import prepare_image, predict_probabilities, compute_saliency_map, make_gradcam_heatmap, save_interpretability_panel

model = keras.models.load_model(MODELS_DIR / "model.keras")
sample_row = splits["test"].iloc[0]

with Image.open(sample_row["path"]) as image:
    image_batch = prepare_image(image, image_size=IMAGE_SIZE)

prediction = predict_probabilities(model, image_batch)
saliency = compute_saliency_map(model, image_batch, prediction["predicted_index"])
gradcam = make_gradcam_heatmap(model, image_batch, prediction["predicted_index"])

save_interpretability_panel(
    image_batch=image_batch,
    saliency=saliency,
    gradcam=gradcam,
    output_path=OUTPUTS_DIR / "interpretability" / "notebook_panel.png",
    title="Interpretabilidad desde notebook",
)

prediction

## 6. App en Streamlit

Ejecuta en terminal desde la raiz del proyecto:

```powershell
streamlit run streamlit_app.py
```

## 7. Entrega final

Revisa la carpeta `docs/` para completar el entregable manuscrito, la guia de despliegue y el checklist.